# 04 - Leakage-safe feature selection (Member 2)

**Goal:** choose, for each feature set (Set 0 / A / B), the subset of features that `05_model_training.ipynb` should use, and save it to `data/processed/selected_features.json`.

**Leakage rules enforced here**

| Rule | How |
|---|---|
| Test rows are never used | only `X_train` / `y_train` from `load_feature_set` are kept; `X_test` is discarded on load |
| Every statistic is learned from training rows | all selectors live in `src/features/select_features.py` and receive training data only |
| Comparison is not optimistic | the **whole** selection is re-run inside each CV training fold; scores come from the held-out fold |
| Non-features are excluded | IDs, survey weights (`WTSAF2YR`...), `split`, target, wear-QC columns are dropped by name/pattern |
| Target-defining labs are blocked | `assert_no_target_leakage` on every candidate column list |

**Pipeline:** non-feature drop -> constant / near-constant drop -> collinearity filter (keeps the member most related to the target) -> **filter** (MI, ANOVA-F with Benjamini-Hochberg, correlation) / **embedded** (RF + GB importance, L1 logistic) / **wrapper** (RFECV) -> vote-based **consensus** -> CV comparison -> pick -> save.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "features").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project (a parent folder must contain src/features/)")
sys.path.insert(0, str(ROOT))

from src.features.data_io import TARGET, assert_no_target_leakage
from src.features.select_features import (
    DEFAULT_SELECTION_PATH, STRATEGIES, SelectionConfig, build_selection_record,
    choose_strategy, compare_strategies_cv, load_selected_features, run_selection, save_selection)
from src.models.pipeline import FEATURE_SET_FILES, load_feature_set

FEATURE_SETS = list(FEATURE_SET_FILES)            # ['Set 0', 'Set A', 'Set B']
CFG = SelectionConfig(random_state=42)            # change thresholds here (see class docstring)
CV_SPLITS, CV_REPEATS = 5, 2                      # 10 held-out evaluations per strategy
STRATEGY_OVERRIDE = {}                            # e.g. {"Set B": "consensus"} to override the automatic pick

FIG_DIR, TABLE_DIR = ROOT / "reports" / "figures", ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True); TABLE_DIR.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE, GREY = "#4C78A8", "#F58518", "#9AA0A6"

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"m2_04_{name}.png", dpi=150, bbox_inches="tight")

print("Project root:", ROOT)
print("Config      :", CFG)

## 2. Load the training split of every feature set

`load_feature_set` applies the fixed stratified 80/20 split created in notebook 03 (`split` column). The test half is **discarded immediately**; nothing below can see it.

In [ ]:
X_train, y_train, rows = {}, {}, []
for s in FEATURE_SETS:
    X, _X_test_discarded, y, _y_test_discarded = load_feature_set(s)
    del _X_test_discarded, _y_test_discarded
    assert_no_target_leakage(X.columns)
    assert TARGET not in X.columns and "split" not in X.columns
    X_train[s], y_train[s] = X, y
    rows.append({"feature_set": s, "n_train": len(X), "n_columns": X.shape[1],
                 "positive_rate": round(y.mean(), 3), "n_missing_cells": int(X.isna().sum().sum())})
display(pd.DataFrame(rows).set_index("feature_set"))

## 3. Fit all selectors on the full training split

One call per feature set runs every stage and returns every candidate subset (`all_clean`, `filter`, `tree`, `l1`, `rfecv`, `consensus`). These are the subsets that get **saved**; section 4 measures how good each *strategy* is without selection bias.

In [ ]:
results = {}
for s in FEATURE_SETS:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        results[s] = run_selection(X_train[s], y_train[s], CFG)
    r = results[s]
    print(f"{s}: {X_train[s].shape[1]} input -> {len(r.candidates)} after cleaning | "
          + " | ".join(f"{k}={len(v)}" for k, v in r.subsets.items() if k != "all_clean"))

stage_table = pd.DataFrame({
    s: {"input columns": X_train[s].shape[1],
        "non-feature dropped": len(r.dropped_non_feature),
        "near-constant dropped": len(r.dropped_near_constant),
        "collinear dropped": len(r.dropped_collinear),
        "after cleaning": len(r.candidates),
        **{f"selected: {k}": len(v) for k, v in r.subsets.items() if k != "all_clean"}}
    for s, r in results.items()}).T
display(stage_table)

### 3a. What the cleaning stages removed

In [ ]:
audit = []
for s, r in results.items():
    audit += [{"feature_set": s, "stage": "non-feature", "feature": f, "detail": ""} for f in r.dropped_non_feature]
    audit += [{"feature_set": s, "stage": "near-constant", "feature": f, "detail": ""} for f in r.dropped_near_constant]
    audit += [{"feature_set": s, "stage": "collinear (|r|>%.2f)" % CFG.corr_threshold, "feature": f,
               "detail": f"redundant with kept '{k}'"} for f, k in r.dropped_collinear.items()]
audit = pd.DataFrame(audit, columns=["feature_set", "stage", "feature", "detail"])
display(audit if len(audit) else "Nothing was dropped by the cleaning stages.")

### 3b. Per-feature evidence from each method family (Set B, the richest set)

In [ ]:
BIG = FEATURE_SETS[-1]
sc = results[BIG].scores
cols = ["feature", "mi", "f_stat", "f_pvalue_bh", "abs_corr_y", "rf_importance", "gb_importance",
        "l1_coef", "votes", "sel_filter", "sel_tree", "sel_l1", "sel_rfecv", "sel_consensus"]
display(sc[cols].round(4))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 5.2), sharey=True)
top = sc.head(20).iloc[::-1]
panels = [("mi", "Mutual information", BLUE), ("f_stat", "ANOVA F statistic", BLUE),
          ("tree_importance", "RF + GB importance (mean, normalised)", ORANGE), ("l1_coef", "L1 logistic coefficient", ORANGE)]
for ax, (col, title, color) in zip(axes, panels):
    ax.barh(top["feature"], top[col], color=color); ax.set_title(title, fontsize=10)
axes[0].set_ylabel(f"{BIG}: top 20 by votes")
fig.suptitle(f"{BIG} - evidence per method (training data only)", y=1.02)
plt.tight_layout(); save_fig(fig, "feature_evidence"); plt.show()

### 3c. RFECV curve (wrapper method)

In [ ]:
fig, axes = plt.subplots(1, len(FEATURE_SETS), figsize=(4.6 * len(FEATURE_SETS), 3.4), sharey=True)
for ax, s in zip(np.atleast_1d(axes), FEATURE_SETS):
    c = results[s].rfecv_curve
    ax.errorbar(c["n_features"], c["mean_auc"], yerr=c["std_auc"], fmt="o-", ms=3, color=BLUE, ecolor=GREY)
    n_sel = len(results[s].subsets["rfecv"])
    ax.axvline(n_sel, color=ORANGE, ls="--", label=f"chosen (1-SE): {n_sel}")
    ax.set(title=s, xlabel="number of features"); ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("inner 5-fold ROC-AUC")
plt.tight_layout(); save_fig(fig, "rfecv_curves"); plt.show()

## 4. Compare strategies with cross-validation (selection inside every fold)

For each CV split the *entire* pipeline above is refitted on the training fold, the resulting subsets are used to train two contrasting learners (logistic regression and random forest), and ROC-AUC / PR-AUC / F1 / precision / recall are measured on the **held-out fold**. Numbers are means over models and 5x2 repeated stratified folds. F1, precision and recall use a 0.5 threshold (both learners are class-balanced); the threshold-free ROC-AUC / PR-AUC are the primary metrics.

In [ ]:
cv = {}
for s in FEATURE_SETS:
    print(f"\n{s}:")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        cv[s] = compare_strategies_cv(X_train[s], y_train[s], CFG, n_splits=CV_SPLITS, n_repeats=CV_REPEATS)
    display(cv[s]["summary"].round(3))

### 4a. Detail per learner

In [ ]:
for s in FEATURE_SETS:
    print(s); display(cv[s]["by_model"].round(3))

### 4b. Metrics by strategy

In [ ]:
fig, axes = plt.subplots(len(FEATURE_SETS), 2, figsize=(13, 3.3 * len(FEATURE_SETS)), squeeze=False)
for row, s in zip(axes, FEATURE_SETS):
    sm = cv[s]["summary"]
    ax = row[0]
    ax.bar(sm.index, sm["roc_auc"], yerr=sm["roc_auc_se"], capsize=3, color=BLUE)
    lo = max(0.5, sm["roc_auc"].min() - 0.03)
    ax.set(ylim=(lo, sm["roc_auc"].max() + 0.02), ylabel="CV ROC-AUC", title=f"{s} - ROC-AUC (+/- SE)")
    for i, (n, v) in enumerate(zip(sm["n_features_mean"], sm["roc_auc"])):
        ax.text(i, lo + 0.002, f"{n:.0f} feat.", ha="center", fontsize=8, color="white")
    ax = row[1]
    x = np.arange(len(sm)); w = 0.2
    for j, (m, c) in enumerate([("pr_auc", BLUE), ("f1", ORANGE), ("precision", "#54A24B"), ("recall", "#B279A2")]):
        ax.bar(x + (j - 1.5) * w, sm[m], w, label=m, color=c)
    ax.set_xticks(x, sm.index); ax.set_ylim(0.4, 0.85); ax.set_title(f"{s} - PR-AUC / F1 / precision / recall")
    if s == FEATURE_SETS[0]: ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); save_fig(fig, "strategy_comparison"); plt.show()

### 4c. Selection stability (share of CV training folds in which a feature was chosen)

In [ ]:
fig, axes = plt.subplots(1, len(FEATURE_SETS), figsize=(5.2 * len(FEATURE_SETS), 7), gridspec_kw={"width_ratios": [1, 1, 1.4]})
for ax, s in zip(axes, FEATURE_SETS):
    st = cv[s]["stability"].drop(columns="all_clean").head(25)
    im = ax.imshow(st.to_numpy(), aspect="auto", cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(st.shape[1]), st.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(st)), st.index, fontsize=8); ax.set_title(s)
fig.colorbar(im, ax=axes, shrink=0.6, label="selection frequency")
save_fig(fig, "selection_stability"); plt.show()
display(pd.concat({s: cv[s]["summary"]["jaccard_stability"] for s in FEATURE_SETS}, axis=1).round(2)
        .rename_axis("strategy").style.set_caption("Mean pairwise Jaccard of fold subsets (1 = identical every fold)"))

## 5. Choose a strategy per feature set

Rule: take the strategy with the best mean CV ROC-AUC, then pick the **smallest** subset whose ROC-AUC is within one standard error of it (one-standard-error rule). Check the stability table above before accepting; use `STRATEGY_OVERRIDE` in the setup cell to force a different one, then re-run from here.

In [ ]:
chosen = {}
for s in FEATURE_SETS:
    chosen[s] = STRATEGY_OVERRIDE.get(s) or choose_strategy(cv[s]["summary"], CFG.one_se_rule)
    assert chosen[s] in STRATEGIES
final = pd.DataFrame({
    s: {"strategy": chosen[s], "n_features": len(results[s].subsets[chosen[s]]),
        "cv_roc_auc": cv[s]["summary"].loc[chosen[s], "roc_auc"],
        "cv_roc_auc_best": cv[s]["summary"]["roc_auc"].max(),
        "cv_f1": cv[s]["summary"].loc[chosen[s], "f1"],
        "jaccard_stability": cv[s]["summary"].loc[chosen[s], "jaccard_stability"]}
    for s in FEATURE_SETS}).T
display(final)
for s in FEATURE_SETS:
    print(f"{s} [{chosen[s]}]:", results[s].subsets[chosen[s]])

## 6. Save outputs for `05_model_training.ipynb`

In [ ]:
records = {s: build_selection_record(chosen[s], results[s], CFG, X_train[s].columns, len(X_train[s]),
                                     cv[s]["summary"].loc[chosen[s]])
           for s in FEATURE_SETS}
path = save_selection(records, DEFAULT_SELECTION_PATH,
                      meta={"random_state": CFG.random_state, "cv_splits": CV_SPLITS, "cv_repeats": CV_REPEATS,
                            "source": "notebooks/04_feature_selection.ipynb"})

# Human-readable tables (all small, tracked in git)
pd.concat({s: cv[s]["summary"] for s in FEATURE_SETS}, names=["feature_set", "strategy"]) \
    .to_csv(TABLE_DIR / "m2_04_strategy_comparison.csv")
pd.concat({s: cv[s]["by_model"] for s in FEATURE_SETS}, names=["feature_set", "strategy", "model"]) \
    .to_csv(TABLE_DIR / "m2_04_strategy_comparison_by_model.csv")
pd.concat({s: results[s].scores for s in FEATURE_SETS}, names=["feature_set", "row"]) \
    .to_csv(TABLE_DIR / "m2_04_feature_scores.csv")
pd.concat({s: cv[s]["stability"] for s in FEATURE_SETS}, names=["feature_set", "feature"]) \
    .to_csv(TABLE_DIR / "m2_04_selection_stability.csv")
final.to_csv(TABLE_DIR / "m2_04_chosen_strategy.csv")
audit.to_csv(TABLE_DIR / "m2_04_dropped_features.csv", index=False)
pd.DataFrame([{"feature_set": s, "strategy": chosen[s], "rank": i + 1, "feature": f}
              for s in FEATURE_SETS for i, f in enumerate(results[s].subsets[chosen[s]])]) \
    .to_csv(TABLE_DIR / "m2_04_selected_features.csv", index=False)

print("selection JSON ->", path)
print("tables         ->", sorted(p.name for p in TABLE_DIR.glob("m2_04_*.csv")))
print("figures        ->", sorted(p.name for p in FIG_DIR.glob("m2_04_*.png")))

## 7. Verification (hard checks) and how notebook 05 loads the result

In [ ]:
for s in FEATURE_SETS:
    feats = load_selected_features(s, available_columns=X_train[s].columns)   # raises on any mismatch
    assert feats == results[s].subsets[chosen[s]]
    assert len(feats) == len(set(feats)) >= 1
    assert set(feats) <= set(X_train[s].columns)
    assert TARGET not in feats and "split" not in feats and not any(f.upper().startswith("WT") for f in feats)
    assert_no_target_leakage(feats)
print("All checks passed: selected features reload cleanly, are unique, exist in the data and contain no target / weights / IDs.")
display(pd.Series({s: len(load_selected_features(s)) for s in FEATURE_SETS}, name="n_selected"))

**In `05_model_training.ipynb`**, replace the `X_train` / `X_test` of each set right after loading:

```python
from src.features.select_features import load_selected_features

for s in FEATURE_SETS:
    X_train, X_test, y_train, y_test = load_feature_set(s)
    cols = load_selected_features(s, available_columns=X_train.columns)
    data[s] = (X_train[cols], X_test[cols], y_train, y_test)
```

The test split is only column-subset, never used to choose columns.